# 02 — Same geometry, different optimization

The PCA subspace is fixed. Only its orthogonal orientation changes. Figure 2
compares downstream outcomes; Figure 3 shows the instantaneous pre/post effect
of each epoch-wise Procrustes refit. Gram error and linear CKA are negative
controls: they must remain invariant under every orthogonal gauge.

In [ ]:
# 1. Settings
from pathlib import Path

REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"
AUTO_PULL_REPO, INSTALL_REQUIREMENTS = True, True
PAIR = "qwen3_4b_to_bert_base"
TRAIN_DATA_REL = Path("data/train_set/train_100k.csv")
RUN_NAME = f"analysis_rotation_{PAIR}_v2"
SEEDS, ROTATION_DRAWS = [42, 43, 44], list(range(10))
BATCH_SIZE, EPOCHS, LR = 128, 5, 7e-5
TARGET_ROWS = 256
EXECUTE, STOP_ON_ERROR, REQUIRE_COMPLETE = False, True, True
CUDA_VISIBLE_DEVICES = "0"

In [ ]:
# 2. Repo, dependencies, GPU và dữ liệu
import subprocess, sys

cwd = Path.cwd().resolve()
PROJECT_DIR = next((p for p in (cwd, cwd.parent) if (p / "main.py").is_file()), None)
if PROJECT_DIR is None:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    assert (PROJECT_DIR / "main.py").is_file(), f"Repo không hợp lệ: {PROJECT_DIR}"

tracked = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "status", "--porcelain", "--untracked-files=no"],
    check=True, capture_output=True, text=True,
).stdout.strip()
if AUTO_PULL_REPO and not tracked:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif AUTO_PULL_REPO:
    print("[git] Bỏ qua pull vì repo có tracked changes.")
if INSTALL_REQUIREMENTS:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
        check=True,
    )

git_head = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
sys.path[:0] = [str(PROJECT_DIR), str(PROJECT_DIR / "notebooks")]
TRAIN_DATA = PROJECT_DIR / TRAIN_DATA_REL
assert TRAIN_DATA.is_file(), f"Thiếu training data: {TRAIN_DATA}"
if EXECUTE:
    import torch
    assert torch.cuda.is_available(), "Hãy bật GPU runtime trước khi train."
print(f"Repo: {PROJECT_DIR} @ {git_head}")
print(f"Training data: {TRAIN_DATA}")

In [ ]:
# 3. Plan: PCA gauge, Haar gauges, fit-once and epoch-wise refit
import shlex
from _analysis_common import PAIRS, collect_jobs, geoode_command, run_jobs

pair, cache_dir = PAIRS[PAIR], PROJECT_DIR / "runs" / "teacher_cache"
run_root = PROJECT_DIR / "runs" / RUN_NAME
run_root.mkdir(parents=True, exist_ok=True)
specs = [
    ("pca_gauge", [None], 0, ["--no-gauge_align"]),
    ("haar", ROTATION_DRAWS, 0, ["--gauge_align", "--gauge_rotation", "random"]),
    ("fit_once", [None], 0, ["--gauge_align", "--gauge_rotation", "procrustes"]),
    ("refit_epoch", [None], 1, ["--gauge_align", "--gauge_rotation", "procrustes"]),
]
jobs = []
for arm, draws, refit, arm_args in specs:
    for draw in draws:
        for seed in SEEDS:
            cell = arm if draw is None else f"{arm}__d{draw}"
            extra = ["--projection_type", "pca", "--lambda_end", 1, "--lambda_ctr", 0,
                     "--lambda_topo", 0, "--gauge_refit_every", refit,
                     "--diag_every", 50, "--no_eval_retrieval", *arm_args]
            if draw is not None:
                extra += ["--gauge_random_seed", draw]
            run_dir = run_root / cell / f"seed_{seed}"
            jobs.append({
                "name": f"{cell}/seed_{seed}", "arm": arm, "draw": draw,
                "seed": seed, "run_dir": run_dir,
                "command": geoode_command(
                    PROJECT_DIR, pair=pair, train_data=TRAIN_DATA, cache_dir=cache_dir,
                    run_dir=run_dir, seed=seed, batch_size=BATCH_SIZE, epochs=EPOCHS,
                    learning_rate=LR, extra=extra,
                ),
            })
print(f"Plan: {len(jobs)} jobs -> {run_root}")
for job in jobs:
    print(shlex.join(job["command"]))

In [ ]:
# 4. Chạy tuần tự; final-test record là resume boundary
from IPython.display import display

if EXECUTE:
    display(run_jobs(
        PROJECT_DIR, jobs, cuda_visible_devices=CUDA_VISIBLE_DEVICES,
        stop_on_error=STOP_ON_ERROR,
    ))
else:
    print("Dry run: đặt EXECUTE=True để chạy các job còn thiếu.")

In [ ]:
# 5. Invariance controls and paper Figures 2–3
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from _analysis_common import load_teacher_cache, set_paper_style, teacher_cache_path
from src import structural_audit as audit

set_paper_style()
results = collect_jobs(jobs)
results.to_csv(run_root / "rotation_by_run.csv", index=False)
done = results.query("status == 'done'").copy()
if done.empty:
    print("No completed runs yet.")
else:
    expected = {"pca_gauge": 3, "haar": 30, "fit_once": 3, "refit_epoch": 3}
    counts = done.groupby("arm").size().to_dict()
    if REQUIRE_COMPLETE and counts != expected:
        raise RuntimeError(f"Incomplete rotation grid: got {counts}, expected {expected}")
    teacher, _ = load_teacher_cache(teacher_cache_path(PROJECT_DIR, cache_dir, pair=pair, train_data=TRAIN_DATA))
    teacher = teacher[:min(TARGET_ROWS, len(teacher))].float()
    def targets(arm, draw=None):
        cell = arm if draw is None else f"{arm}__d{draw}"
        path = run_root / cell / f"seed_{SEEDS[0]}" / "teacher_projection.pt"
        return audit.targets_from_saved(teacher, torch.load(path, map_location="cpu", weights_only=False))
    reference = targets("pca_gauge")
    variants = {"PCA gauge": reference, "Fit once": targets("fit_once"),
                "Refit / epoch": targets("refit_epoch"), "Haar Q": targets("haar", ROTATION_DRAWS[0])}
    invariance = pd.DataFrame([
        {"variant": name, "gram_rmse": audit.gram_rmse(reference, value),
         "linear_cka": audit.linear_cka(reference, value)}
        for name, value in variants.items()
    ])
    invariance.to_csv(run_root / "rotation_invariance.csv", index=False)
    display(invariance.style.format({"gram_rmse": "{:.3e}", "linear_cka": "{:.6f}"}))

    order = ["pca_gauge", "haar", "fit_once", "refit_epoch"]
    labels = ["PCA gauge", "Haar rotations", "Fit once", "Refit / epoch"]
    fig, ax = plt.subplots(figsize=(5.5, 2.45))
    random_scores = 100 * done.query("arm == 'haar'")["avg_all"].dropna().to_numpy()
    box = ax.boxplot([random_scores], positions=[1], widths=.48, patch_artist=True, showfliers=False)
    box["boxes"][0].set(facecolor="#D4D8DE", edgecolor="#6B7280")
    for item in box["medians"] + box["whiskers"] + box["caps"]:
        item.set(color="#1F2937", linewidth=1.2)
    colors = {"pca_gauge": "#6B7280", "fit_once": "#2B6CB0", "refit_epoch": "#DD6B20"}
    for position, arm in enumerate(order):
        if arm == "haar":
            continue
        values = 100 * done.query("arm == @arm")["avg_all"].dropna()
        ax.errorbar(position, values.mean(), yerr=values.std(ddof=1), fmt="o", ms=6,
                    capsize=3, color=colors[arm], zorder=3)
    ax.set(xticks=range(4), xticklabels=labels, ylabel="Final average score")
    ax.text(.02, .97, f"CKA ≥ {invariance.linear_cka.min():.6f}\nmax Gram RMSE = {invariance.gram_rmse.max():.1e}",
            transform=ax.transAxes, va="top", fontsize=7, color="#4B5563")
    fig.tight_layout()
    fig.savefig(run_root / "figure_2_same_geometry_different_kd.pdf", bbox_inches="tight")
    fig.savefig(run_root / "figure_2_same_geometry_different_kd.png", dpi=300, bbox_inches="tight")
    plt.show()

    events = []
    for job in jobs:
        if job["arm"] != "refit_epoch":
            continue
        saved = torch.load(Path(job["run_dir"]) / "teacher_projection.pt", map_location="cpu", weights_only=False)
        for event in saved.get("gauge_history", []):
            if "epoch" not in event:
                continue
            events += [
                {"seed": job["seed"], "epoch": event["epoch"], "state": "Before refit", "error": 1 - event["cos_previous_gauge"]},
                {"seed": job["seed"], "epoch": event["epoch"], "state": "After refit", "error": 1 - event["cos_after"]},
            ]
    events = pd.DataFrame(events)
    events.to_csv(run_root / "gauge_refit_events.csv", index=False)
    stats = events.groupby(["state", "epoch"])["error"].agg(["mean", "std"]).reset_index()
    fig, ax = plt.subplots(figsize=(5.5, 2.45))
    offsets = {"Before refit": -.055, "After refit": .055}
    colors = {"Before refit": "#6B7280", "After refit": "#DD6B20"}
    for state in offsets:
        part = stats.query("state == @state").sort_values("epoch")
        ax.errorbar(part.epoch + offsets[state], part["mean"], yerr=part["std"].fillna(0),
                    marker="o", capsize=2.5, color=colors[state], label=state)
    for epoch in sorted(events.epoch.unique()):
        pair_values = stats.query("epoch == @epoch").set_index("state")["mean"]
        ax.plot([epoch + offsets[s] for s in offsets], [pair_values[s] for s in offsets],
                color="#C7CBD1", lw=1, zorder=0)
    ax.set(xlabel="Epoch boundary", ylabel="Frozen-subset alignment error ↓",
           xticks=sorted(events.epoch.unique()))
    ax.legend(frameon=False, ncol=2)
    fig.tight_layout()
    fig.savefig(run_root / "figure_3_refit_restores_interface.pdf", bbox_inches="tight")
    fig.savefig(run_root / "figure_3_refit_restores_interface.png", dpi=300, bbox_inches="tight")
    plt.show()